# Used Vehicle Price Analysis

**Goals:**
- **Understand** what makes a car expensive or cheap — and communicate it to a non-technical audience
- **Predict** the listing price of a car given its attributes
- **Segment** the market into buyer groups to guide inventory and pricing decisions

---
## Phase 1 — Business Understanding

**Business problem:** A used-car marketplace needs to understand what drives listing price so it can set competitive prices, guide buyers, and optimise inventory.

**Three concrete questions:**
1. **What makes a car expensive or cheap?** (interpretability for non-technical stakeholders)
2. **Can we predict listing price from a vehicle's attributes?** (regression)
3. **Are there natural buyer groups we can target differently?** (clustering / segmentation)

**Success criteria:**
- A model that explains > 80 % of price variation (R² ≥ 0.80)
- A ranked list of top price drivers confirmed by two independent methods
- Actionable segment profiles (price range, age, mileage per segment)

---
## Phase 2 — Data Understanding

Load the raw dataset, audit quality issues, and explore patterns before any cleaning or modelling.

### 2.1  Load the Data

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from sklearn.linear_model import LinearRegression, Ridge, Lasso, LassoCV, RidgeCV
from sklearn.preprocessing import StandardScaler, OneHotEncoder, PolynomialFeatures
from sklearn.model_selection import train_test_split, cross_val_score, KFold, GridSearchCV
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error
from sklearn.inspection import permutation_importance
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.metrics import silhouette_score

sns.set_theme(style='whitegrid')
plt.rcParams['figure.figsize'] = (12, 5)
RANDOM_STATE = 42
CURRENT_YEAR = 2024

In [ ]:
df = pd.read_csv('data/vehicles.csv')
print('Raw shape:', df.shape)
df.head(3)

In [ ]:
# Missing values
missing = (df.isnull().sum() / len(df) * 100).round(1).sort_values(ascending=False)
print('Missing % per column:')
print(missing[missing > 0].to_string())

### 2.2  Exploratory Data Analysis

Inspect distributions and relationships to understand which features matter and how they relate to price.

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(16, 10))

axes[0,0].hist(df['price'], bins=60, color='steelblue', edgecolor='white')
axes[0,0].set_title('Price Distribution (Raw)')
axes[0,0].set_xlabel('Price ($)')

axes[0,1].hist(df['log_price'], bins=60, color='darkorange', edgecolor='white')
axes[0,1].set_title('Log Price Distribution')
axes[0,1].set_xlabel('Log Price')

axes[0,2].scatter(df['vehicle_age'], df['price'], alpha=0.05, s=4, color='teal')
axes[0,2].set_title('Vehicle Age vs Price')
axes[0,2].set_xlabel('Vehicle Age (years)')
axes[0,2].set_ylabel('Price ($)')

axes[1,0].scatter(df['log_odometer'], df['log_price'], alpha=0.05, s=4, color='purple')
axes[1,0].set_title('Log Odometer vs Log Price')
axes[1,0].set_xlabel('Log Odometer')
axes[1,0].set_ylabel('Log Price')

cond_price = df.groupby('condition')['price'].median().sort_values(ascending=False)
axes[1,1].bar(cond_price.index, cond_price.values, color='steelblue', edgecolor='white')
axes[1,1].set_title('Median Price by Condition')
axes[1,1].set_xlabel('Condition')
axes[1,1].set_ylabel('Median Price ($)')
axes[1,1].tick_params(axis='x', rotation=30)

top_mfr = df.groupby('manufacturer')['price'].median().sort_values(ascending=False).head(10)
axes[1,2].barh(top_mfr.index, top_mfr.values, color='darkorange', edgecolor='white')
axes[1,2].set_title('Top 10 Manufacturers by Median Price')
axes[1,2].set_xlabel('Median Price ($)')

plt.suptitle('Exploratory Analysis — Used Vehicle Listings', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('images/eda_overview.png', dpi=150)
plt.show()

In [ ]:
# Correlation heatmap
num_cols = ['price', 'log_price', 'vehicle_age', 'log_odometer', 'mileage_per_year']
corr = df[num_cols].corr()

plt.figure(figsize=(8, 6))
sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm', center=0,
            square=True, linewidths=0.5)
plt.title('Feature Correlation Matrix')
plt.tight_layout()
plt.savefig('images/correlation_heatmap.png', dpi=150)
plt.show()

In [ ]:
# Price depreciation curve by age
age_price = df.groupby('vehicle_age')['price'].median().reset_index()

plt.figure(figsize=(12, 5))
plt.plot(age_price['vehicle_age'], age_price['price'], 'o-', color='steelblue', linewidth=2)
plt.fill_between(age_price['vehicle_age'], age_price['price'], alpha=0.15, color='steelblue')
plt.xlabel('Vehicle Age (years)')
plt.ylabel('Median Listing Price ($)')
plt.title('Depreciation Curve — Median Price by Vehicle Age')
plt.tight_layout()
plt.savefig('images/depreciation_curve.png', dpi=150)
plt.show()

### 2.3  Price Trend Analysis (by Model Year)

No listing-date column is available, so **model year** is used as the time axis.
We test stationarity with the ADF test, fit ARIMA, and overlay a linear trend.

In [ ]:
from statsmodels.tsa.arima.model import ARIMA
from statsmodels.tsa.stattools import adfuller
from scipy import stats as scipy_stats

# Aggregate to annual median price by model year
yearly = (
    df
    .groupby('year')['price']
    .agg(['median', 'count'])
    .rename(columns={'median': 'median_price', 'count': 'listing_count'})
    .query('listing_count >= 50')
    .reset_index()
    .sort_values('year')
)

print(f'{len(yearly)} model years of data')
print(f'Year range: {int(yearly["year"].min())} — {int(yearly["year"].max())}')
yearly

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(14, 8), sharex=True)

axes[0].plot(yearly['year'], yearly['median_price'], 'o-', color='steelblue', linewidth=2)
axes[0].fill_between(yearly['year'], yearly['median_price'], alpha=0.15, color='steelblue')
axes[0].set_title('Median Listing Price by Model Year')
axes[0].set_ylabel('Median Price ($)')

axes[1].bar(yearly['year'], yearly['listing_count'], color='darkorange', width=0.6)
axes[1].set_title('Listing Volume by Model Year')
axes[1].set_ylabel('Listings')
axes[1].set_xlabel('Model Year')

plt.suptitle('Used Car Market — Price & Volume by Model Year', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('images/price_trend.png', dpi=150)
plt.show()

In [ ]:
series = yearly.set_index('year')['median_price']

adf_stat, p_val, _, _, crit_vals, _ = adfuller(series.dropna())
stationary = p_val < 0.05

print(f'ADF Statistic  : {adf_stat:.4f}')
print(f'p-value        : {p_val:.4f}')
print(f'Critical values: {crit_vals}')
print(f'Stationary     : {"Yes" if stationary else "No — will use d=1 (first difference)"}')

fig, axes = plt.subplots(1, 2, figsize=(14, 4))
axes[0].plot(series, 'o-', color='steelblue', linewidth=2)
axes[0].set_title('Original Series (by Model Year)')
axes[0].set_xlabel('Model Year')
axes[1].plot(series.diff().dropna(), 'o-', color='darkorange', linewidth=2)
axes[1].set_title('First Difference (year-over-year change)')
axes[1].set_xlabel('Model Year')
plt.tight_layout()
plt.savefig('images/stationarity_check.png', dpi=150)
plt.show()

In [ ]:
d = 0 if stationary else 1
order = (1, d, 1)

fitted = ARIMA(series, order=order).fit()

N_FORECAST = 3
forecast   = fitted.get_forecast(steps=N_FORECAST)
fc_mean    = forecast.predicted_mean
fc_ci      = forecast.conf_int()

plt.figure(figsize=(14, 5))
plt.plot(series, 'o-', label='Observed', color='steelblue', linewidth=2)
plt.plot(fc_mean, 's--', label=f'Forecast (+{N_FORECAST} model years)', color='crimson', linewidth=2)
plt.fill_between(fc_ci.index, fc_ci.iloc[:, 0], fc_ci.iloc[:, 1],
                 alpha=0.25, color='crimson', label='95% CI')
plt.title(f'ARIMA{order} — Median Price Forecast by Model Year')
plt.ylabel('Median Price ($)')
plt.xlabel('Model Year')
plt.legend()
plt.tight_layout()
plt.savefig('images/arima_forecast.png', dpi=150)
plt.show()

print('Forecast (next 3 model years):')
print(pd.DataFrame({'Forecast ($)': fc_mean.round(0),
                    'Lower 95%':    fc_ci.iloc[:, 0].round(0),
                    'Upper 95%':    fc_ci.iloc[:, 1].round(0)}))

In [ ]:
x = np.arange(len(series))
slope, intercept, r, p_val_trend, _ = scipy_stats.linregress(x, series.values)
trend_line = intercept + slope * x

plt.figure(figsize=(14, 5))
plt.plot(series.index, series.values, 'o-', label='Observed', color='steelblue', linewidth=2)
plt.plot(series.index, trend_line, '--',
         label=f'Linear Trend ({slope:+.0f}/year)', color='darkorange', linewidth=2)
plt.title('Used Car Price — Linear Trend by Model Year')
plt.ylabel('Median Price ($)')
plt.xlabel('Model Year')
plt.legend()
plt.tight_layout()
plt.savefig('images/price_linear_trend.png', dpi=150)
plt.show()

print(f'Slope   : ${slope:+.2f} per model year')
print(f'R²      : {r**2:.4f}')
print(f'p-value : {p_val_trend:.4f}  ({"significant" if p_val_trend < 0.05 else "not significant"})')

---
## Phase 3 — Data Preparation

Clean the raw listings and engineer features that a model can use.

### 3.1  Data Cleaning

Raw listings contain junk entries and extensive missing values.
Each problem is handled separately rather than using a blanket `dropna()`.

- Columns with **> 60 % missing** are dropped entirely
- Rows missing `price`, `year`, or `odometer` are dropped (model backbone)
- Other categorical columns are **imputed with the mode** to preserve rows
- `price = 0` entries are removed as invalid/placeholder
- **IQR-based bounds** replace hardcoded limits so the data decides thresholds

In [ ]:
# ── Step 1: Drop unhelpful or near-empty columns ──
drop_always = ['id', 'VIN', 'region', 'url', 'region_url', 'image_url',
               'description', 'county', 'lat', 'long']
df = df.drop(columns=[c for c in drop_always if c in df.columns], errors='ignore')

missing_pct = df.isnull().sum() / len(df) * 100
high_missing = missing_pct[missing_pct > 60].index.tolist()
print(f'Dropping {len(high_missing)} columns with >60% missing: {high_missing}')
df = df.drop(columns=high_missing, errors='ignore')

# ── Step 2: Handle price = 0 ──
n_zero = (df['price'] == 0).sum()
print(f'\nRecords with price = 0: {n_zero:,}  →  removed (treated as invalid)')
df = df[df['price'] > 0]

# ── Step 3: Drop rows only where essential columns are missing ──
essential = ['price', 'year', 'odometer']
before = len(df)
df = df.dropna(subset=essential)
print(f'Dropped {before - len(df):,} rows missing essential columns {essential}')
print(f'Remaining: {len(df):,} rows')

# ── Step 4: Impute non-essential categorical columns with mode ──
cat_cols = ['manufacturer', 'fuel', 'transmission', 'condition', 'title_status',
            'drive', 'type', 'paint_color', 'state']
print('\nImputing categorical columns:')
for col in cat_cols:
    if col in df.columns:
        n_missing = df[col].isnull().sum()
        if n_missing > 0:
            mode_val = df[col].mode()[0]
            df[col] = df[col].fillna(mode_val)
            print(f'  {col:15s}: {n_missing:,} filled with mode = "{mode_val}"')

# ── Step 5: IQR-based outlier removal ──
def iqr_bounds(series, factor=1.5):
    Q1  = series.quantile(0.25)
    Q3  = series.quantile(0.75)
    IQR = Q3 - Q1
    return Q1 - factor * IQR, Q3 + factor * IQR

p_low, p_high = iqr_bounds(df['price'])
p_low  = max(p_low,  500)
p_high = min(p_high, 150_000)
print(f'\nPrice bounds    : ${p_low:,.0f}  —  ${p_high:,.0f}')

y_low, y_high = iqr_bounds(df['year'])
y_low  = max(y_low,  1980)
y_high = min(y_high, 2023)
print(f'Year bounds     :  {y_low:.0f}  —  {y_high:.0f}')

o_low, o_high = iqr_bounds(df['odometer'])
o_low  = max(o_low, 0)
o_high = min(o_high, 500_000)
print(f'Odometer bounds :  {o_low:,.0f}  —  {o_high:,.0f} miles')

df = df[(df['price']    >= p_low)  & (df['price']    <= p_high)]
df = df[(df['year']     >= y_low)  & (df['year']     <= y_high)]
df = df[(df['odometer'] >= o_low)  & (df['odometer'] <= o_high)]

print(f'\nCleaned shape: {df.shape}')

### 3.2  Feature Engineering

| Transformation | Rationale |
|---|---|
| `year` → `vehicle_age` | Depreciation is driven by age, not calendar year |
| `odometer` → `log_odometer` | Mileage–price relationship is log-linear |
| `price` → `log_price` | Symmetric, well-behaved regression target |
| `mileage_per_year` | Usage intensity, independent of age |
| Ordered label encoding | `condition`, `title_status` have a natural order |
| Target encoding | `manufacturer`, `state` — replace with mean log-price (avoids sparse dummies) |
| One-hot encoding | `fuel`, `transmission`, `drive`, `type`, `paint_color` |

In [ ]:
# ── Numeric engineered features ──
df['vehicle_age']      = CURRENT_YEAR - df['year']
df['mileage_per_year'] = df['odometer'] / (df['vehicle_age'].replace(0, 1))
df['log_price']        = np.log1p(df['price'])
df['log_odometer']     = np.log1p(df['odometer'])

# ── Ordered categoricals → label encoding (order is meaningful) ──
ordered_cols = {
    'condition':    ['salvage', 'fair', 'good', 'excellent', 'like new', 'new'],
    'title_status': ['missing', 'parts only', 'salvage', 'rebuilt', 'lien', 'clean'],
}
for col, order in ordered_cols.items():
    if col in df.columns:
        df[col] = df[col].astype(str).str.lower().str.strip()
        df[col + '_enc'] = df[col].map(
            {v: i for i, v in enumerate(order)}
        ).fillna(len(order))

# ── Target encoding for high-cardinality categoricals ──
# Replaces each category with the mean log_price of that group.
# Avoids the sparse dummy-column explosion from one-hot encoding high-cardinality variables.
global_mean_lp = df['log_price'].mean()
high_card_cols = [c for c in ['manufacturer', 'state'] if c in df.columns]
te_features = []
for col in high_card_cols:
    te_map = df.groupby(col)['log_price'].mean()
    df[col + '_te'] = df[col].map(te_map).fillna(global_mean_lp)
    te_features.append(col + '_te')
    print(f'Target-encoded "{col}": {df[col].nunique()} categories → 1 column')

# ── One-hot encoding for low-cardinality categoricals ──
onehot_cols = [c for c in ['fuel', 'transmission', 'drive', 'type', 'paint_color']
               if c in df.columns]
ohe = OneHotEncoder(sparse_output=False, handle_unknown='ignore', dtype=np.int8, drop='first')
ohe_array = ohe.fit_transform(df[onehot_cols].astype(str))
ohe_names = ohe.get_feature_names_out(onehot_cols).tolist()
ohe_df    = pd.DataFrame(ohe_array, columns=ohe_names, index=df.index)
df        = pd.concat([df, ohe_df], axis=1)

# ── Final feature list ──
numeric_features = ['vehicle_age', 'log_odometer', 'mileage_per_year']
label_features   = [c + '_enc' for c in ordered_cols if c + '_enc' in df.columns]
feature_cols     = numeric_features + label_features + te_features + ohe_names

print(f'\nNumeric features  : {len(numeric_features)}')
print(f'Label-encoded     : {len(label_features)}  {label_features}')
print(f'Target-encoded    : {len(te_features)}  {te_features}')
print(f'One-hot features  : {len(ohe_names)}')
print(f'Total features    : {len(feature_cols)}')

---
## Phase 4 — Modeling

Train regression models to predict price and a clustering model to discover market segments.

### 4.1  Price Prediction Models

Three regularised regression models are compared:
- **Linear Regression** — baseline; draws a straight line through the data
- **Ridge** — L2 regularisation; keeps all features but shrinks weak ones
- **Lasso** — L1 regularisation; zeros out weak features (built-in selection)

**Evaluation:** 5-fold cross-validation + held-out test set (R², RMSE, MAE).
All features are standardised (0 mean, unit variance) before fitting.

In [ ]:
from matplotlib.patches import FancyBboxPatch

def _workflow(steps, title, palette, filename):
    n = len(steps)
    fig, ax = plt.subplots(figsize=(12, max(6, n * 1.05 + 1.0)))
    ax.set_xlim(0, 10)
    ax.set_ylim(0, n * 1.05 + 0.4)
    ax.axis('off')
    BOX_H, BOX_W, STEP_H = 0.62, 8.6, 1.05
    for i, (label, detail) in enumerate(steps):
        y = (n - i) * STEP_H - 0.25
        color = palette[i % len(palette)]
        ax.add_patch(FancyBboxPatch(
            (0.7, y - BOX_H / 2), BOX_W, BOX_H,
            boxstyle="round,pad=0.08", fc=color, ec="white", lw=1.5, alpha=0.92))
        ax.text(5, y + (0.10 if detail else 0), f"{i+1}.  {label}",
                ha="center", va="center", fontsize=10, fontweight="bold", color="white")
        if detail:
            ax.text(5, y - 0.20, detail,
                    ha="center", va="center", fontsize=8.5, color="white", alpha=0.88)
        if i < n - 1:
            ay0 = y - BOX_H / 2 - 0.03
            ax.annotate("", xy=(5, ay0 - (STEP_H - BOX_H) + 0.10), xytext=(5, ay0),
                        arrowprops=dict(arrowstyle="->", color="#555", lw=1.8))
    ax.set_title(title, fontsize=13, fontweight="bold", pad=14)
    plt.tight_layout()
    plt.savefig(filename, dpi=150, bbox_inches="tight")
    plt.show()

_P = ["#2171b5","#2171b5","#238b45","#6a3d9a","#6a3d9a","#6a3d9a",
      "#d6604d","#d6604d","#c7843e","#c7843e","#1a9850"]

_workflow([
    ("Load Feature Matrix",
     f"{len(feature_cols)} features — numeric | ordered label-enc | target-enc | one-hot"),
    ("Train / Test Split  80 / 20",
     "stratified random split  |  random_state=42  |  test set never touched during training"),
    ("StandardScaler",
     "fit on train only → transform train & test  |  zero mean, unit variance"),
    ("Fit Linear Regression  (baseline)",
     "no regularisation — establishes the performance floor"),
    ("Fit Ridge  (RidgeCV)",
     "L2 penalty — shrinks weak coefficients but keeps all features"),
    ("Fit Lasso  (LassoCV)",
     "L1 penalty — zeros out weak features  →  built-in feature selection"),
    ("5-Fold Cross-Validation",
     "R² across 5 folds — confirms results are not split-dependent"),
    ("Hyperparameter Tuning  (GridSearchCV)",
     "exhaustive alpha grid  |  plots R² sensitivity vs regularisation strength"),
    ("Feature Analysis",
     "Lasso |coef| ranking  +  permutation importance (model-agnostic, shuffle-based)"),
    ("Polynomial Features  (degree=2)",
     "expand 3 numeric features → 9 terms  |  captures curvature & interactions"),
    ("Model Leaderboard",
     "rank all models by Test R², RMSE, MAE"),
], "Phase 4.1 — Price Prediction Model Workflow", _P,
   "images/workflow_4_1_regression.png")


In [ ]:
X = df[feature_cols].fillna(0)
y = df['log_price']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE
)

scaler = StandardScaler()
X_train_sc = scaler.fit_transform(X_train)
X_test_sc  = scaler.transform(X_test)

print(f'Train: {X_train_sc.shape}  |  Test: {X_test_sc.shape}')

In [ ]:
def evaluate(name, model, X_tr, X_te, y_tr, y_te):
    ytr = model.predict(X_tr)
    yte = model.predict(X_te)
    return {
        'Model':      name,
        'Train R²':   round(r2_score(y_tr, ytr), 4),
        'Test R²':    round(r2_score(y_te, yte), 4),
        'Test RMSE':  round(np.sqrt(mean_squared_error(y_te, yte)), 4),
        'Test MAE':   round(mean_absolute_error(y_te, yte), 4)
    }

results = []

# ── Linear Regression (baseline) ──
lr = LinearRegression().fit(X_train_sc, y_train)
results.append(evaluate('Linear Regression', lr, X_train_sc, X_test_sc, y_train, y_test))

# ── Ridge (auto alpha via CV) ──
ridge = RidgeCV(alphas=np.logspace(-3, 5, 30), cv=3)
ridge.fit(X_train_sc, y_train)
results.append(evaluate(f'Ridge (α={ridge.alpha_:.3f})', ridge, X_train_sc, X_test_sc, y_train, y_test))
print(f'Ridge best alpha: {ridge.alpha_:.4f}')

# ── Lasso (auto alpha via CV) ──
# Search from 10⁻⁴ to 10¹ — shifted up so Lasso actually cuts weak features
SAMPLE = min(30_000, len(X_train_sc))
idx = np.random.RandomState(RANDOM_STATE).choice(len(X_train_sc), size=SAMPLE, replace=False)

lasso_search = LassoCV(
    alphas=np.logspace(-4, 1, 30),
    cv=3,
    max_iter=3_000,
    n_jobs=-1
)
lasso_search.fit(X_train_sc[idx], y_train.iloc[idx])
best_alpha = lasso_search.alpha_

lasso = Lasso(alpha=best_alpha, max_iter=5_000)
lasso.fit(X_train_sc, y_train)

results.append(evaluate(f'Lasso (α={best_alpha:.5f})', lasso, X_train_sc, X_test_sc, y_train, y_test))
print(f'Lasso best alpha   : {best_alpha:.6f}')
print(f'Lasso features kept: {np.sum(lasso.coef_ != 0)} / {len(feature_cols)}')
print(f'Lasso features cut : {np.sum(lasso.coef_ == 0)} / {len(feature_cols)}')

pd.DataFrame(results)

In [ ]:
# Cross-validation scores
kf = KFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
cv_models = {
    'Linear Regression': LinearRegression(),
    'Ridge':             Ridge(alpha=ridge.alpha_),
    'Lasso':             Lasso(alpha=lasso.alpha, max_iter=10_000)
}

cv_scores = {}
for name, model in cv_models.items():
    scores = cross_val_score(model, X_train_sc, y_train, cv=kf, scoring='r2', n_jobs=-1)
    cv_scores[name] = scores
    print(f'{name:22s} | CV R² = {scores.mean():.4f} ± {scores.std():.4f}')

plt.figure(figsize=(9, 5))
pd.DataFrame(cv_scores).boxplot()
plt.ylabel('R² per Fold')
plt.title('5-Fold Cross-Validation R² — Model Comparison')
plt.tight_layout()
plt.savefig('images/cv_comparison.png', dpi=150)
plt.show()

#### 4.1a  Hyperparameter Tuning — Grid Search

`GridSearchCV` records CV scores for every alpha, letting us plot sensitivity and compare Ridge vs Lasso directly.

In [ ]:
alphas_ridge = np.logspace(-2, 4, 25)
alphas_lasso = np.logspace(-4, 1, 25)

# ── Ridge grid search ──
grid_ridge = GridSearchCV(
    Ridge(), {'alpha': alphas_ridge},
    cv=5, scoring='r2', n_jobs=-1, refit=True
)
grid_ridge.fit(X_train_sc, y_train)

# ── Lasso grid search ──
grid_lasso = GridSearchCV(
    Lasso(max_iter=5_000), {'alpha': alphas_lasso},
    cv=5, scoring='r2', n_jobs=-1, refit=True
)
grid_lasso.fit(X_train_sc, y_train)

print(f'Ridge — best alpha: {grid_ridge.best_params_["alpha"]:.4f}  |  '
      f'CV R²: {grid_ridge.best_score_:.4f}  |  '
      f'Test R²: {r2_score(y_test, grid_ridge.predict(X_test_sc)):.4f}')
print(f'Lasso — best alpha: {grid_lasso.best_params_["alpha"]:.4f}  |  '
      f'CV R²: {grid_lasso.best_score_:.4f}  |  '
      f'Test R²: {r2_score(y_test, grid_lasso.predict(X_test_sc)):.4f}')

# ── Plot CV R² vs alpha for both models ──
ridge_res = pd.DataFrame(grid_ridge.cv_results_)
lasso_res = pd.DataFrame(grid_lasso.cv_results_)

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

for ax, res, alphas, best, label, color in [
    (axes[0], ridge_res, alphas_ridge, grid_ridge.best_params_['alpha'], 'Ridge', 'steelblue'),
    (axes[1], lasso_res, alphas_lasso, grid_lasso.best_params_['alpha'], 'Lasso', 'darkorange'),
]:
    ax.semilogx(alphas, res['mean_test_score'], 'o-', color=color)
    ax.fill_between(alphas,
                    res['mean_test_score'] - res['std_test_score'],
                    res['mean_test_score'] + res['std_test_score'],
                    alpha=0.2, color=color)
    ax.axvline(best, color='crimson', linestyle='--', label=f'Best α = {best:.4f}')
    ax.set_xlabel('Alpha (log scale)')
    ax.set_ylabel('CV R²')
    ax.set_title(f'{label} — CV R² vs Alpha')
    ax.legend()

plt.suptitle('Grid Search: Ridge vs Lasso Hyperparameter Sensitivity', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('images/ridge_grid_search.png', dpi=150)
plt.show()

In [ ]:
# Lasso feature importance
coef_df = pd.DataFrame({'Feature': feature_cols, 'Coefficient': lasso.coef_})
coef_df = coef_df[coef_df['Coefficient'] != 0].sort_values('Coefficient', key=abs, ascending=False)

plt.figure(figsize=(11, max(5, len(coef_df) * 0.4)))
colors = ['steelblue' if c > 0 else 'crimson' for c in coef_df['Coefficient']]
plt.barh(coef_df['Feature'], coef_df['Coefficient'], color=colors, edgecolor='white')
plt.axvline(0, color='black', linewidth=0.8)
plt.xlabel('Lasso Coefficient (standardized)')
plt.title('Lasso — Selected Features & Their Impact on Price')
plt.tight_layout()
plt.savefig('images/lasso_features.png', dpi=150)
plt.show()

#### 4.1b  Permutation Importance

A model-agnostic check: shuffle one feature at a time and measure the R² drop.
Large drop → the model genuinely relied on that feature.
Near-zero → redundant or replaceable.

In [ ]:
# Use a sample to keep runtime manageable
SAMPLE_PERM = min(10_000, len(X_test_sc))
idx_perm = np.random.RandomState(RANDOM_STATE).choice(len(X_test_sc), size=SAMPLE_PERM, replace=False)

perm_result = permutation_importance(
    lasso,
    X_test_sc[idx_perm],
    y_test.iloc[idx_perm],
    n_repeats=10,
    random_state=RANDOM_STATE,
    n_jobs=-1
)

perm_df = pd.DataFrame({
    'Feature':    feature_cols,
    'Importance': perm_result.importances_mean,
    'Std':        perm_result.importances_std
}).sort_values('Importance', ascending=False).head(20)

plt.figure(figsize=(10, 6))
colors = ['steelblue' if v >= 0 else 'lightgray' for v in perm_df['Importance']]
plt.barh(perm_df['Feature'], perm_df['Importance'],
         xerr=perm_df['Std'], color=colors, edgecolor='white')
plt.axvline(0, color='black', linewidth=0.8)
plt.xlabel('Mean decrease in R² when feature is shuffled')
plt.title('Permutation Importance — Top 20 Features (Lasso Model)')
plt.tight_layout()
plt.savefig('images/permutation_importance.png', dpi=150)
plt.show()

print('\nTop 10 features by permutation importance:')
print(perm_df.head(10)[['Feature', 'Importance', 'Std']].to_string(index=False))

In [ ]:
# Best model: Predicted vs Actual
best_pred = lasso.predict(X_test_sc)
residuals = y_test - best_pred

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].scatter(y_test, best_pred, alpha=0.15, s=5, color='steelblue')
mn, mx = y_test.min(), y_test.max()
axes[0].plot([mn, mx], [mn, mx], 'r--', linewidth=1.5)
axes[0].set_xlabel('Actual Log Price')
axes[0].set_ylabel('Predicted Log Price')
axes[0].set_title(f'Predicted vs Actual  (R²={r2_score(y_test, best_pred):.4f})')

axes[1].hist(residuals, bins=60, color='darkorange', edgecolor='white')
axes[1].axvline(0, color='red', linewidth=1)
axes[1].set_xlabel('Residual')
axes[1].set_title('Residual Distribution')

plt.suptitle('Lasso Model — Prediction Quality', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('images/lasso_prediction.png', dpi=150)
plt.show()

In [ ]:
# Final leaderboard
leaderboard = pd.DataFrame(results).sort_values('Test R²', ascending=False).reset_index(drop=True)
print('=== Model Leaderboard ===')
leaderboard

#### 4.1c  Polynomial Features

A degree-2 expansion of the three core numeric features adds squared terms and cross-products,
letting the linear model capture curvature and interactions without switching algorithms.

In [ ]:
core_idx = list(range(len(numeric_features)))  # first N columns are numeric
poly = PolynomialFeatures(degree=2, include_bias=False)

X_poly_tr = poly.fit_transform(X_train_sc[:, core_idx])
X_poly_te = poly.transform(X_test_sc[:, core_idx])

X_train_poly = np.hstack([X_poly_tr, X_train_sc[:, len(core_idx):]])
X_test_poly  = np.hstack([X_poly_te,  X_test_sc[:, len(core_idx):]])

poly_feature_names = poly.get_feature_names_out(numeric_features).tolist()
print(f'Polynomial features added : {len(poly_feature_names)}  (from {len(numeric_features)} numeric inputs)')
print(f'New feature matrix shape  : {X_train_poly.shape}')
print(f'New features: {poly_feature_names}')

ridge_poly = RidgeCV(alphas=np.logspace(-3, 5, 30), cv=3)
ridge_poly.fit(X_train_poly, y_train)

base_r2 = r2_score(y_test, ridge.predict(X_test_sc))
poly_r2 = r2_score(y_test, ridge_poly.predict(X_test_poly))

print(f'\nRidge (baseline)           Test R² = {base_r2:.4f}')
print(f'Ridge + Polynomial (deg=2) Test R² = {poly_r2:.4f}')
print(f'Improvement                        : {(poly_r2 - base_r2)*100:+.2f} percentage points')

### 4.2  Market Segmentation (K-Means)

K-Means discovers natural buyer groups — budget, everyday, premium — without being told what those groups are.

Clustering uses 4 interpretable features (`log_price`, `vehicle_age`, `log_odometer`, `mileage_per_year`).
PCA reduces to 2D only for the scatter plot; grouping uses all 4 features.

In [ ]:
from matplotlib.patches import FancyBboxPatch

def _workflow(steps, title, palette, filename):
    n = len(steps)
    fig, ax = plt.subplots(figsize=(12, max(6, n * 1.05 + 1.0)))
    ax.set_xlim(0, 10)
    ax.set_ylim(0, n * 1.05 + 0.4)
    ax.axis('off')
    BOX_H, BOX_W, STEP_H = 0.62, 8.6, 1.05
    for i, (label, detail) in enumerate(steps):
        y = (n - i) * STEP_H - 0.25
        color = palette[i % len(palette)]
        ax.add_patch(FancyBboxPatch(
            (0.7, y - BOX_H / 2), BOX_W, BOX_H,
            boxstyle="round,pad=0.08", fc=color, ec="white", lw=1.5, alpha=0.92))
        ax.text(5, y + (0.10 if detail else 0), f"{i+1}.  {label}",
                ha="center", va="center", fontsize=10, fontweight="bold", color="white")
        if detail:
            ax.text(5, y - 0.20, detail,
                    ha="center", va="center", fontsize=8.5, color="white", alpha=0.88)
        if i < n - 1:
            ay0 = y - BOX_H / 2 - 0.03
            ax.annotate("", xy=(5, ay0 - (STEP_H - BOX_H) + 0.10), xytext=(5, ay0),
                        arrowprops=dict(arrowstyle="->", color="#555", lw=1.8))
    ax.set_title(title, fontsize=13, fontweight="bold", pad=14)
    plt.tight_layout()
    plt.savefig(filename, dpi=150, bbox_inches="tight")
    plt.show()

_P = ["#1d6996","#1d6996","#0f7554","#0f7554","#edad08","#edad08",
      "#cc503e","#cc503e","#94346e"]

_workflow([
    ("Select 4 Clustering Features",
     "log_price  |  vehicle_age  |  log_odometer  |  mileage_per_year"),
    ("StandardScaler",
     "distance-based algorithm — all features must be on the same scale"),
    ("Sample 20 K rows  (speed)",
     "silhouette evaluation is O(n²)  |  full data used for final fit"),
    ("Try k = 2 … 9",
     "fit KMeans(n_init=10) for each k  |  record inertia + silhouette score"),
    ("Select Best k",
     "argmax silhouette score  |  cross-check with elbow inflection point"),
    ("Fit Final KMeans  (all data)",
     f"k = BEST_K  |  n_init=10  |  assign segment label to every listing in df"),
    ("PCA  4D → 2D",
     "scatter-plot only  |  explained variance shown  |  clustering used full 4D"),
    ("Segment Profiles",
     "groupby segment → mean price, age, odometer, mileage/yr, listing count"),
    ("Boxplots per Segment",
     "distribution of price, vehicle age, odometer across all segments"),
], "Phase 4.2 — Market Segmentation Workflow", _P,
   "images/workflow_4_2_kmeans.png")


In [ ]:
cluster_features = ['log_price', 'vehicle_age', 'log_odometer', 'mileage_per_year']
X_cluster = StandardScaler().fit_transform(df[cluster_features].fillna(0))

SAMPLE = min(20_000, len(X_cluster))
idx_s  = np.random.RandomState(RANDOM_STATE).choice(len(X_cluster), size=SAMPLE, replace=False)
X_s    = X_cluster[idx_s]

k_range    = range(2, 10)
inertias   = []
silhouettes= []

for k in k_range:
    km  = KMeans(n_clusters=k, random_state=RANDOM_STATE, n_init=10)
    lbl = km.fit_predict(X_s)
    inertias.append(km.inertia_)
    silhouettes.append(silhouette_score(X_s, lbl, sample_size=5000))

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].plot(k_range, inertias,    'o-', color='steelblue')
axes[0].set_xlabel('k'); axes[0].set_ylabel('Inertia'); axes[0].set_title('Elbow Method')
axes[1].plot(k_range, silhouettes, 's-', color='darkorange')
axes[1].set_xlabel('k'); axes[1].set_ylabel('Silhouette Score'); axes[1].set_title('Silhouette Score')
plt.tight_layout()
plt.savefig('images/kmeans_k_selection.png', dpi=150)
plt.show()

BEST_K = list(k_range)[np.argmax(silhouettes)]
print(f'Best k = {BEST_K}')

In [ ]:
km_final = KMeans(n_clusters=BEST_K, random_state=RANDOM_STATE, n_init=10)
df['segment'] = km_final.fit_predict(X_cluster)

pca = PCA(n_components=2, random_state=RANDOM_STATE)
X_2d = pca.fit_transform(X_cluster)

plt.figure(figsize=(11, 7))
colors = plt.cm.tab10(np.linspace(0, 1, BEST_K))
for k in range(BEST_K):
    mask = df['segment'] == k
    s_idx = np.where(mask)[0]
    s_idx = s_idx[np.random.choice(len(s_idx), size=min(3000, len(s_idx)), replace=False)]
    plt.scatter(X_2d[s_idx, 0], X_2d[s_idx, 1],
                s=8, alpha=0.4, color=colors[k], label=f'Segment {k}')

centroids_2d = pca.transform(km_final.cluster_centers_)
plt.scatter(centroids_2d[:, 0], centroids_2d[:, 1],
            s=200, marker='X', color='black', zorder=10, label='Centroids')
plt.xlabel('PC1'); plt.ylabel('PC2')
plt.title(f'K-Means Market Segments (k={BEST_K})')
plt.legend(markerscale=2)
plt.tight_layout()
plt.savefig('images/market_segments.png', dpi=150)
plt.show()

In [ ]:
profile = df.groupby('segment')[['price', 'vehicle_age', 'odometer', 'mileage_per_year']].mean().round(0)
profile['listing_count'] = df.groupby('segment').size()
profile = profile.sort_values('price', ascending=False)
print('=== Market Segment Profiles ===')
profile

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

df.boxplot(column='price',       by='segment', ax=axes[0], grid=False)
df.boxplot(column='vehicle_age', by='segment', ax=axes[1], grid=False)
df.boxplot(column='odometer',    by='segment', ax=axes[2], grid=False)

axes[0].set_title('Price by Segment');       axes[0].set_xlabel('Segment')
axes[1].set_title('Vehicle Age by Segment'); axes[1].set_xlabel('Segment')
axes[2].set_title('Odometer by Segment');    axes[2].set_xlabel('Segment')
[fig.suptitle('') for _ in [1]]

plt.suptitle('Market Segment Characteristics', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('images/segment_profiles.png', dpi=150)
plt.show()

### 4.3  Time-Series Modeling & Forecasting

Use vehicle model year as the time axis to build a formal time-series model of used-car prices.

**Workflow:** Build TS → Decompose → Test stationarity → ACF/PACF → ARMA order selection → ARIMA grid search → Residual diagnostics → Forecast → SARIMA → Walk-forward validation → Depreciation rate → Model comparison

In [ ]:
from matplotlib.patches import FancyBboxPatch

def _workflow(steps, title, palette, filename):
    n = len(steps)
    fig, ax = plt.subplots(figsize=(12, max(6, n * 1.05 + 1.0)))
    ax.set_xlim(0, 10)
    ax.set_ylim(0, n * 1.05 + 0.4)
    ax.axis('off')
    BOX_H, BOX_W, STEP_H = 0.62, 8.6, 1.05
    for i, (label, detail) in enumerate(steps):
        y = (n - i) * STEP_H - 0.25
        color = palette[i % len(palette)]
        ax.add_patch(FancyBboxPatch(
            (0.7, y - BOX_H / 2), BOX_W, BOX_H,
            boxstyle="round,pad=0.08", fc=color, ec="white", lw=1.5, alpha=0.92))
        ax.text(5, y + (0.10 if detail else 0), f"{i+1}.  {label}",
                ha="center", va="center", fontsize=10, fontweight="bold", color="white")
        if detail:
            ax.text(5, y - 0.20, detail,
                    ha="center", va="center", fontsize=8.5, color="white", alpha=0.88)
        if i < n - 1:
            ay0 = y - BOX_H / 2 - 0.03
            ax.annotate("", xy=(5, ay0 - (STEP_H - BOX_H) + 0.10), xytext=(5, ay0),
                        arrowprops=dict(arrowstyle="->", color="#555", lw=1.8))
    ax.set_title(title, fontsize=13, fontweight="bold", pad=14)
    plt.tight_layout()
    plt.savefig(filename, dpi=150, bbox_inches="tight")
    plt.show()

_P = ["#355f8d","#355f8d","#277f8e","#277f8e","#1fa187","#1fa187",
      "#4ac16d","#4ac16d","#a0da39","#a0da39","#fde725","#fde725"]

_workflow([
    ("Build Time Series",
     "mean price per model year  |  filter years with ≥ 50 listings  |  annual frequency"),
    ("Decompose  (period = 5)",
     "Additive decomposition  +  STL (robust)  →  trend, seasonal, residual components"),
    ("Stationarity Tests",
     "ADF (H₀: unit root)  +  KPSS (H₀: stationary)  on raw, 1st-diff, log+1st-diff"),
    ("ACF / PACF",
     "identify candidate AR(p) and MA(q) orders before grid search"),
    ("ARMA Grid Search  (stationary series)",
     "7 candidate orders on 1st-difference  |  rank by AIC, BIC, test RMSE"),
    ("ARIMA Grid Search  (original series)",
     "p ∈ [0,3], d ∈ [0,2], q ∈ [0,3]  →  up to 48 models  |  best by AIC"),
    ("Residual Diagnostics",
     "ACF of residuals  |  Q-Q plot  |  histogram vs normal  →  check white noise"),
    ("Forecast  + 95% CI",
     "10-step ahead forecast with confidence band  |  test RMSE and MAPE reported"),
    ("SARIMA Grid Search  (S = 5)",
     "5 non-seasonal × 5 seasonal orders  →  25 candidates  |  captures 5-yr cycles"),
    ("Walk-Forward Validation",
     "expanding window, 1-step ahead  |  most realistic estimate for production"),
    ("Depreciation Rate Analysis",
     "YoY % change per model year  |  3-yr and 5-yr rolling means"),
    ("Model Comparison",
     "ARIMA vs SARIMA vs Walk-Forward  |  RMSE ($) and MAPE (%)"),
], "Phase 4.3 — Time-Series Modeling Workflow", _P,
   "images/workflow_4_3_timeseries.png")


In [ ]:
from statsmodels.tsa.stattools import adfuller, kpss
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf
from statsmodels.tsa.seasonal import seasonal_decompose, STL
from statsmodels.tsa.arima.model import ARIMA
from statsmodels.tsa.statespace.sarimax import SARIMAX
from statsmodels.graphics.gofplots import qqplot
from scipy.stats import norm as sp_norm

In [ ]:
# Build annual time series from the already-cleaned df
ts_raw = (
    df.groupby('year')
    .agg(mean_price=('price', 'mean'), count=('price', 'count'))
    .reset_index()
)
ts_raw = ts_raw[ts_raw['count'] >= 50].copy()
ts_raw = ts_raw.sort_values('year').set_index('year')

ts = ts_raw['mean_price'].astype(float)
ts.index = pd.to_datetime(ts.index, format='%Y')
ts.index.freq = 'AS'

print(f'Time series length: {len(ts)} years')
print(ts.describe().round(0))

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(14, 8))

axes[0].plot(ts.index.year, ts.values, 'o-', color='steelblue', linewidth=2)
axes[0].fill_between(ts.index.year, ts.values, alpha=0.15, color='steelblue')
axes[0].set_xlabel('Vehicle Model Year')
axes[0].set_ylabel('Mean Listing Price ($)')
axes[0].set_title('Mean Used Car Listing Price by Vehicle Model Year')

axes[1].bar(ts_raw.index.year, ts_raw['count'], color='darkorange', edgecolor='white')
axes[1].set_xlabel('Vehicle Model Year')
axes[1].set_ylabel('Number of Listings')
axes[1].set_title('Listing Count by Model Year')

plt.tight_layout()
plt.savefig('images/ts_raw.png', dpi=150)
plt.show()

#### 4.3a  Decomposition — Trend, Seasonality, Residual

Three decomposition methods separate the series into interpretable components.
**Additive** assumes components sum; **Multiplicative** assumes they multiply;
**STL** is robust to outliers and handles non-linear trends.

In [ ]:
decomp_add = seasonal_decompose(ts, model='additive', period=5)

fig, axes = plt.subplots(4, 1, figsize=(14, 12))
for ax, comp, label, col in zip(
    axes,
    [decomp_add.observed, decomp_add.trend, decomp_add.seasonal, decomp_add.resid],
    ['Observed', 'Trend', 'Seasonal (period=5)', 'Residual'],
    ['steelblue', 'darkorange', 'green', 'crimson']
):
    ax.plot(ts.index.year, comp, color=col, linewidth=1.5)
    if label == 'Residual':
        ax.axhline(0, color='black', linewidth=0.8, linestyle='--')
    ax.set_title(label); ax.set_xlabel('Year')

plt.suptitle('Classical Additive Decomposition', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('images/ts_decompose_additive.png', dpi=150)
plt.show()

In [ ]:
stl = STL(ts, period=5, robust=True)
stl_result = stl.fit()

fig, axes = plt.subplots(4, 1, figsize=(14, 12))
for ax, comp, label, col in zip(
    axes,
    [stl_result.observed, stl_result.trend, stl_result.seasonal, stl_result.resid],
    ['Observed', 'Trend (STL)', 'Seasonal (STL, period=5)', 'Residual (STL)'],
    ['steelblue', 'darkorange', 'green', 'crimson']
):
    ax.plot(ts.index.year, comp, color=col, linewidth=1.5)
    if 'Residual' in label:
        ax.axhline(0, color='black', linewidth=0.8, linestyle='--')
    ax.set_title(label); ax.set_xlabel('Year')

plt.suptitle('STL Decomposition (Robust)', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('images/ts_decompose_stl.png', dpi=150)
plt.show()

#### 4.3b  Stationarity Tests

Most time-series models require a stationary series (constant mean and variance).

- **ADF test** (H₀: unit root present → non-stationary) — reject H₀ to conclude stationary
- **KPSS test** (H₀: series is stationary) — fail to reject H₀ to conclude stationary

Both tests together give a more robust conclusion than either alone.

In [ ]:
def adf_test(series, label=''):
    r = adfuller(series.dropna(), autolag='AIC')
    print(f'\n--- ADF Test: {label} ---')
    print(f'  Statistic: {r[0]:.4f}  |  p-value: {r[1]:.4f}')
    print(f'  Conclusion: {"STATIONARY" if r[1] < 0.05 else "NON-STATIONARY"}')
    return r[1]

def kpss_test(series, label=''):
    r = kpss(series.dropna(), regression='c', nlags='auto')
    print(f'\n--- KPSS Test: {label} ---')
    print(f'  Statistic: {r[0]:.4f}  |  p-value: {r[1]:.4f}')
    print(f'  Conclusion: {"NON-STATIONARY" if r[1] < 0.05 else "STATIONARY"}')
    return r[1]

adf_p  = adf_test(ts,  'Raw Series')
kpss_p = kpss_test(ts, 'Raw Series')

In [ ]:
ts_diff1    = ts.diff().dropna()
ts_log      = np.log(ts)
ts_log_diff = ts_log.diff().dropna()

adf_test(ts_diff1,    '1st Difference')
kpss_test(ts_diff1,   '1st Difference')
adf_test(ts_log_diff, 'Log + 1st Diff')
kpss_test(ts_log_diff,'Log + 1st Diff')

In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(14, 12))

for ax, s, title, col in zip(
    axes,
    [ts, ts_diff1, ts_log_diff],
    ['Original Series (non-stationary)', '1st Difference', 'Log + 1st Difference'],
    ['steelblue', 'darkorange', 'green']
):
    ax.plot(s.index.year, s.values, 'o-', color=col, linewidth=1.5)
    ax.axhline(0, color='black', linewidth=0.8, linestyle='--')
    ax.set_title(title); ax.set_xlabel('Year')

plt.suptitle('Stationarity Transformations', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('images/ts_stationarity.png', dpi=150)
plt.show()

#### 4.3c  ACF & PACF — Identifying ARMA Order

| Pattern | Implies |
|---|---|
| ACF cuts off at lag q, PACF tails off | MA(q) |
| PACF cuts off at lag p, ACF tails off | AR(p) |
| Both tail off | ARMA(p, q) |

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(16, 10))

plot_acf( ts,       lags=min(20, len(ts)//2-1),       ax=axes[0,0], title='ACF — Original')
plot_pacf(ts,       lags=min(20, len(ts)//2-1),       ax=axes[0,1], title='PACF — Original', method='ywm')
plot_acf( ts_diff1, lags=min(15, len(ts_diff1)//2-1), ax=axes[1,0], title='ACF — 1st Difference')
plot_pacf(ts_diff1, lags=min(15, len(ts_diff1)//2-1), ax=axes[1,1], title='PACF — 1st Difference', method='ywm')

plt.suptitle('Autocorrelation Analysis for ARMA Order Selection', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('images/ts_acf_pacf.png', dpi=150)
plt.show()

#### 4.3d  ARMA Model Selection

Fit candidate ARMA models on the stationary (1st-difference) series and rank by AIC/BIC.
Hold out the last 5 years as a test window to measure out-of-sample RMSE.

In [ ]:
TS_HOLDOUT = 5
train_diff = ts_diff1.iloc[:-TS_HOLDOUT]
test_diff  = ts_diff1.iloc[-TS_HOLDOUT:]

arma_candidates = [(1,0,0),(0,0,1),(1,0,1),(2,0,0),(2,0,1),(1,0,2),(2,0,2)]
arma_results = []

for arma_order in arma_candidates:
    try:
        fit = ARIMA(train_diff, order=arma_order).fit()
        fc  = fit.forecast(steps=TS_HOLDOUT)
        rmse = np.sqrt(np.mean((fc.values - test_diff.values)**2))
        arma_results.append({'Order': str(arma_order), 'AIC': round(fit.aic, 2),
                             'BIC': round(fit.bic, 2), 'RMSE (test)': round(rmse, 2)})
    except Exception as e:
        print(f'ARMA{order}: FAILED — {e}')

arma_df = pd.DataFrame(arma_results).sort_values('AIC')
print('=== ARMA Model Ranking by AIC ===')
print(arma_df.to_string(index=False))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

arma_plot = arma_df.set_index('Order')
arma_plot['AIC'].plot(kind='bar', ax=axes[0], color='steelblue', edgecolor='white')
axes[0].set_title('AIC by ARMA Order (lower = better)')
axes[0].tick_params(axis='x', rotation=30)

arma_plot['BIC'].plot(kind='bar', ax=axes[1], color='darkorange', edgecolor='white')
axes[1].set_title('BIC by ARMA Order (lower = better)')
axes[1].tick_params(axis='x', rotation=30)

plt.suptitle('ARMA Model Selection: AIC / BIC', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('images/ts_arma_aic_bic.png', dpi=150)
plt.show()

#### 4.3e  ARIMA — Grid Search on Original Series

Search all combinations of (p, d, q) where p ∈ [0,3], d ∈ [0,2], q ∈ [0,3].
`d` is chosen automatically by AIC rather than fixed from the ADF result,
giving the grid search full flexibility.

In [ ]:
train_orig = ts.iloc[:-TS_HOLDOUT]
test_orig  = ts.iloc[-TS_HOLDOUT:]

best_arima_aic = np.inf
best_arima_order = None
arima_results = []

for p in range(0, 4):
    for d in range(0, 3):
        for q in range(0, 4):
            try:
                fit = ARIMA(train_orig, order=(p, d, q)).fit()
                fc  = fit.forecast(steps=TS_HOLDOUT)
                rmse = np.sqrt(np.mean((fc.values - test_orig.values)**2))
                arima_results.append({'p':p,'d':d,'q':q,
                                      'AIC':round(fit.aic,2),'BIC':round(fit.bic,2),
                                      'RMSE':round(rmse,2)})
                if fit.aic < best_arima_aic:
                    best_arima_aic   = fit.aic
                    best_arima_order = (p, d, q)
                    best_arima_fit   = fit
            except:
                pass

arima_df = pd.DataFrame(arima_results).sort_values('AIC')
print(f'Best ARIMA order by AIC: {best_arima_order}  (AIC={best_arima_aic:.2f})')
print('\nTop 10 models:')
print(arima_df.head(10).to_string(index=False))

#### 4.3f  Best ARIMA — Diagnostics & Residual Analysis

Well-fitted residuals should look like white noise:
- No autocorrelation (ACF within bands)
- Roughly normal distribution
- Q-Q plot close to the diagonal

In [ ]:
print(best_arima_fit.summary())

In [ ]:
arima_resid = best_arima_fit.resid

fig, axes = plt.subplots(2, 2, figsize=(14, 10))

axes[0,0].plot(train_orig.index.year, arima_resid, 'o-', color='steelblue', linewidth=1)
axes[0,0].axhline(0, color='red', linewidth=0.8, linestyle='--')
axes[0,0].set_title('Residuals Over Time'); axes[0,0].set_xlabel('Year')

axes[0,1].hist(arima_resid, bins=20, color='darkorange', edgecolor='white', density=True)
xr = np.linspace(arima_resid.min(), arima_resid.max(), 100)
axes[0,1].plot(xr, sp_norm.pdf(xr, arima_resid.mean(), arima_resid.std()), 'r-', linewidth=2)
axes[0,1].set_title('Residual Distribution')

plot_acf(arima_resid.dropna(), lags=min(15, len(residuals)//2-1), ax=axes[1,0],
         title='ACF of Residuals (should be white noise)')

qqplot(arima_resid.dropna(), line='s', ax=axes[1,1])
axes[1,1].set_title('Q-Q Plot of Residuals')

plt.suptitle(f'ARIMA{best_arima_order} — Residual Diagnostics', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('images/ts_arima_diagnostics.png', dpi=150)
plt.show()

#### 4.3g  Forecasting with Confidence Intervals

In [ ]:
FORECAST_STEPS = TS_HOLDOUT + 5

fc_result = best_arima_fit.get_forecast(steps=FORECAST_STEPS)
fc_mean   = fc_result.predicted_mean
fc_ci     = fc_result.conf_int(alpha=0.05)

last_year = train_orig.index[-1].year
fc_years  = np.arange(last_year + 1, last_year + 1 + FORECAST_STEPS)

plt.figure(figsize=(14, 7))
plt.plot(ts.index.year, ts.values, 'o-', color='steelblue', linewidth=2, label='Observed')
plt.plot(fc_years, fc_mean.values, 's--', color='darkorange', linewidth=2, label='Forecast')
plt.fill_between(fc_years, fc_ci.iloc[:,0], fc_ci.iloc[:,1],
                 color='darkorange', alpha=0.2, label='95% CI')
plt.scatter(test_orig.index.year, test_orig.values, color='crimson', zorder=5, s=60, label='Actual (test)')
plt.axvline(last_year, color='gray', linestyle=':', linewidth=1.2, label='Train/Test split')
plt.xlabel('Vehicle Model Year'); plt.ylabel('Mean Listing Price ($)')
plt.title(f'ARIMA{best_arima_order} Forecast with 95% Confidence Intervals')
plt.legend()
plt.tight_layout()
plt.savefig('images/ts_forecast.png', dpi=150)
plt.show()

test_fc   = fc_mean.values[:TS_HOLDOUT]
rmse_test = np.sqrt(np.mean((test_fc - test_orig.values)**2))
mape_test = np.mean(np.abs((test_fc - test_orig.values) / test_orig.values)) * 100
print(f'Test RMSE: ${rmse_test:,.0f}')
print(f'Test MAPE: {mape_test:.2f}%')

#### 4.3h  SARIMA — Seasonal ARIMA

Extends ARIMA with seasonal (P, D, Q) terms at period S = 5,
capturing 5-year depreciation cycles visible in the decomposition.

In [ ]:
S = 5
best_sarima_aic   = np.inf
best_sarima_order = None
sarima_results    = []

for p, d, q in [(1,1,1),(1,1,0),(0,1,1),(2,1,1),(1,1,2)]:
    for P, D, Q in [(0,0,0),(1,0,0),(0,0,1),(1,0,1),(1,1,0)]:
        try:
            fit = SARIMAX(train_orig, order=(p,d,q), seasonal_order=(P,D,Q,S),
                          enforce_stationarity=False, enforce_invertibility=False).fit(disp=False)
            fc   = fit.forecast(steps=TS_HOLDOUT)
            rmse = np.sqrt(np.mean((fc.values - test_orig.values)**2))
            sarima_results.append({'Order': f'({p},{d},{q})x({P},{D},{Q},{S})',
                                   'AIC': round(fit.aic,2), 'BIC': round(fit.bic,2),
                                   'RMSE (test)': round(rmse,2)})
            if fit.aic < best_sarima_aic:
                best_sarima_aic   = fit.aic
                best_sarima_order = ((p,d,q),(P,D,Q,S))
                best_sarima_fit   = fit
        except:
            pass

sarima_df = pd.DataFrame(sarima_results).sort_values('AIC')
print(f'Best SARIMA: {best_sarima_order}  (AIC={best_sarima_aic:.2f})')
print('\nTop 10 SARIMA models:')
print(sarima_df.head(10).to_string(index=False))

In [ ]:
sarima_fc   = best_sarima_fit.get_forecast(steps=FORECAST_STEPS)
sarima_mean = sarima_fc.predicted_mean
sarima_ci   = sarima_fc.conf_int(alpha=0.05)

plt.figure(figsize=(14, 7))
plt.plot(ts.index.year, ts.values, 'o-', color='steelblue', linewidth=2, label='Observed')
plt.plot(fc_years, fc_mean.values,     's--', color='darkorange', linewidth=1.5,
         label=f'ARIMA{best_arima_order}')
plt.plot(fc_years, sarima_mean.values, 'D--', color='green',      linewidth=1.5,
         label=f'SARIMA{best_sarima_order[0]}x{best_sarima_order[1]}')
plt.fill_between(fc_years, sarima_ci.iloc[:,0], sarima_ci.iloc[:,1],
                 color='green', alpha=0.15, label='SARIMA 95% CI')
plt.scatter(test_orig.index.year, test_orig.values, color='crimson', zorder=5, s=60, label='Actual (test)')
plt.axvline(last_year, color='gray', linestyle=':', linewidth=1.2)
plt.xlabel('Vehicle Model Year'); plt.ylabel('Mean Listing Price ($)')
plt.title('ARIMA vs SARIMA Forecast Comparison')
plt.legend()
plt.tight_layout()
plt.savefig('images/ts_sarima_comparison.png', dpi=150)
plt.show()

#### 4.3i  Walk-Forward Validation

More realistic than a single train/test split: re-trains on an expanding window
and forecasts 1 step ahead, simulating how the model would perform in production.

In [ ]:
min_train   = len(ts) - TS_HOLDOUT
walk_preds, walk_actuals, walk_years = [], [], []

for i in range(TS_HOLDOUT):
    window = ts.iloc[:min_train + i]
    actual = ts.iloc[min_train + i]
    try:
        pred = ARIMA(window, order=best_arima_order).fit().forecast(steps=1).values[0]
    except:
        pred = window.iloc[-1]
    walk_preds.append(pred)
    walk_actuals.append(actual)
    walk_years.append(ts.index[min_train + i].year)

walk_rmse = np.sqrt(np.mean((np.array(walk_preds) - np.array(walk_actuals))**2))
walk_mape = np.mean(np.abs((np.array(walk_preds) - np.array(walk_actuals)) / np.array(walk_actuals))) * 100

plt.figure(figsize=(12, 5))
plt.plot(ts.index.year, ts.values, 'o-', color='steelblue', linewidth=2, label='Observed')
plt.plot(walk_years, walk_preds, 's--', color='crimson', linewidth=2, label='Walk-Forward Forecast')
plt.xlabel('Year'); plt.ylabel('Mean Listing Price ($)')
plt.title(f'Walk-Forward Validation — ARIMA{best_arima_order}  |  RMSE=${walk_rmse:,.0f}  MAPE={walk_mape:.1f}%')
plt.legend()
plt.tight_layout()
plt.savefig('images/ts_walk_forward.png', dpi=150)
plt.show()

print(f'Walk-Forward RMSE: ${walk_rmse:,.0f}')
print(f'Walk-Forward MAPE: {walk_mape:.2f}%')

#### 4.3j  Depreciation Rate Analysis

Year-over-year % price change derived from the time series, plus rolling means to smooth noise.

In [ ]:
ts_pct = ts.pct_change().dropna() * 100

fig, axes = plt.subplots(1, 2, figsize=(16, 5))

axes[0].bar(ts_pct.index.year, ts_pct.values,
            color=['crimson' if v < 0 else 'steelblue' for v in ts_pct.values],
            edgecolor='white')
axes[0].axhline(0, color='black', linewidth=0.8)
axes[0].axhline(ts_pct.mean(), color='darkorange', linestyle='--', linewidth=1.5,
                label=f'Mean {ts_pct.mean():.1f}%/yr')
axes[0].set_xlabel('Model Year'); axes[0].set_ylabel('YoY Price Change (%)')
axes[0].set_title('Year-over-Year Price Change by Model Year')
axes[0].legend()

roll3 = ts.rolling(3).mean()
roll5 = ts.rolling(5).mean()
axes[1].plot(ts.index.year, ts.values,    'o-', color='steelblue',  alpha=0.5, linewidth=1, label='Raw')
axes[1].plot(ts.index.year, roll3.values, '-',  color='darkorange', linewidth=2, label='3-yr Rolling Mean')
axes[1].plot(ts.index.year, roll5.values, '-',  color='crimson',    linewidth=2, label='5-yr Rolling Mean')
axes[1].set_xlabel('Model Year'); axes[1].set_ylabel('Mean Price ($)')
axes[1].set_title('Rolling Mean Price')
axes[1].legend()

plt.tight_layout()
plt.savefig('images/ts_depreciation.png', dpi=150)
plt.show()

#### 4.3k  Time-Series Model Comparison

In [ ]:
sarima_test_fc = best_sarima_fit.forecast(steps=TS_HOLDOUT).values
sarima_rmse = np.sqrt(np.mean((sarima_test_fc - test_orig.values)**2))
sarima_mape = np.mean(np.abs((sarima_test_fc - test_orig.values) / test_orig.values)) * 100

ts_comparison = pd.DataFrame([
    {'Model': f'ARIMA{best_arima_order}',
     'Test RMSE': f'${rmse_test:,.0f}', 'Test MAPE': f'{mape_test:.2f}%'},
    {'Model': f'SARIMA{best_sarima_order}',
     'Test RMSE': f'${sarima_rmse:,.0f}', 'Test MAPE': f'{sarima_mape:.2f}%'},
    {'Model': 'Walk-Forward ARIMA',
     'Test RMSE': f'${walk_rmse:,.0f}', 'Test MAPE': f'{walk_mape:.2f}%'},
])
print('=== Time-Series Model Comparison ===')
print(ts_comparison.to_string(index=False))

---
## Phase 5 — Evaluation

Assess whether the models meet the business success criteria defined in Phase 1
and surface actionable insights for stakeholders.

### 5.1  Key Findings

Evaluate results against the three success criteria defined in Phase 1, then surface the most actionable takeaways from each modelling phase.

**Phase 1 criteria recap:**
- R² ≥ 0.80 on held-out data
- Top price drivers confirmed by two independent methods (Lasso coefficients + permutation importance)
- Actionable segment profiles (price, age, mileage per segment)

In [ ]:
best_model_row = leaderboard.iloc[0]
best_r2 = best_model_row['Test R²']

print('=' * 60)
print('  PHASE 5 — EVALUATION: KEY FINDINGS')
print('=' * 60)

# ── Business criteria check ──────────────────────────────────────────────
print('\n BUSINESS CRITERIA CHECK  (Phase 1 targets)')

# Criterion 1: R² ≥ 0.80
met1 = best_r2 >= 0.80
print(f'   Criterion 1  R² ≥ 0.80         :  {best_r2:.4f}  →  {"✓ MET" if met1 else "✗ NOT MET"}')

# Criterion 2: top drivers confirmed by two independent methods
top_lasso = set(coef_df.head(5)['Feature'].tolist())
top_perm  = set(perm_df.head(5)['Feature'].tolist())
overlap   = top_lasso & top_perm
met2 = len(overlap) >= 3
print(f'   Criterion 2  Two-method agreement:  {len(overlap)}/5 top features in both Lasso & permutation importance  →  {"✓ MET" if met2 else "✗ NOT MET"}')
print(f'                Shared: {sorted(overlap)}')

# Criterion 3: actionable segment profiles
met3 = len(profile) >= 2
print(f'   Criterion 3  Segment profiles    :  {len(profile)} segments with price/age/mileage stats  →  {"✓ MET" if met3 else "✗ NOT MET"}')

all_met = met1 and met2 and met3
print(f'\n   Overall: {"All criteria met ✓" if all_met else "Some criteria not met — see below"}')

# ── Best model summary ───────────────────────────────────────────────────
print('\n BEST REGRESSION MODEL  (Phase 4.1)')
print(f'   {best_model_row["Model"]}')
print(f'   Test R²  : {best_r2:.4f}  ({best_r2*100:.1f}% of price variation explained)')
print(f'   Test RMSE: {best_model_row["Test RMSE"]:.4f} log-price units')
approx_pct = (np.exp(best_model_row['Test RMSE']) - 1) * 100
print(f'             ≈ {approx_pct:.0f}% typical relative price error  (stakeholder interpretation)')
print(f'   Unexplained {(1-best_r2)*100:.1f}%: negotiation, hidden damage, local demand')

# ── Takeaway 1: Depreciation ─────────────────────────────────────────────
median_0_5  = df[df['vehicle_age'] <= 5]['price'].median()
median_6_10 = df[(df['vehicle_age'] > 5) & (df['vehicle_age'] <= 10)]['price'].median()
pct_drop    = (median_0_5 - median_6_10) / median_0_5 * 100
print(f'\n   Takeaway 1 — Depreciation steepest in first 5 years')
print(f'      0–5 yr median  : ${median_0_5:,.0f}')
print(f'      6–10 yr median : ${median_6_10:,.0f}  ({pct_drop:.1f}% lower)')

# ── Takeaway 2: mileage_per_year vs raw odometer ─────────────────────────
corr_odo = df['odometer'].corr(df['price'])
corr_mpy = df['mileage_per_year'].corr(df['price'])
winner   = 'mileage_per_year' if abs(corr_mpy) > abs(corr_odo) else 'odometer'
print(f'\n   Takeaway 2 — {winner} is the stronger price signal')
print(f'      odometer vs price     : r = {corr_odo:.3f}')
print(f'      mileage/year vs price : r = {corr_mpy:.3f}')

# ── Takeaway 3: Lasso feature selection ──────────────────────────────────
n_ohe_total = len(ohe_names)
ohe_mask    = [f in ohe_names for f in feature_cols]
ohe_coefs   = lasso.coef_[[i for i, m in enumerate(ohe_mask) if m]]
n_ohe_kept  = int((ohe_coefs != 0).sum())
n_ohe_cut   = n_ohe_total - n_ohe_kept
print(f'\n   Takeaway 3 — Lasso zeroed {n_ohe_cut}/{n_ohe_total} one-hot features ({n_ohe_cut/n_ohe_total*100:.0f}%)')
print(f'      Kept {n_ohe_kept} one-hot features. Top 5 drivers by |coef|:')
for feat in coef_df.head(5)['Feature'].tolist():
    print(f'      • {feat}')

# ── Takeaway 4: Segment volume ────────────────────────────────────────────
largest_seg   = profile['listing_count'].idxmax()
premium_seg   = profile['price'].idxmax()
budget_seg    = profile['price'].idxmin()
budget_count  = int(profile.loc[budget_seg,  'listing_count'])
premium_count = int(profile.loc[premium_seg, 'listing_count'])
vol_ratio     = budget_count / premium_count
print(f'\n   Takeaway 4 — Budget segment dominates inventory ({vol_ratio:.1f}x more listings than premium)')
print(f'      Budget : {budget_count:,} listings  avg ${profile.loc[budget_seg, "price"]:,.0f}')
print(f'      Premium: {premium_count:,} listings  avg ${profile.loc[premium_seg, "price"]:,.0f}')

# ── Takeaway 5: Segment price spread ─────────────────────────────────────
price_gap = profile.loc[premium_seg, 'price'] - profile.loc[budget_seg, 'price']
print(f'\n   Takeaway 5 — ${price_gap:,.0f} price gap between premium and budget')
for seg, row in profile.iterrows():
    print(f'      Segment {seg}: ${row["price"]:,.0f}  |  {row["vehicle_age"]:.0f} yr  |  {row["odometer"]:,.0f} mi  |  {int(row["listing_count"]):,} listings')

# ── Takeaway 6: Time-series forecast ─────────────────────────────────────
direction_word = 'rising' if slope > 0 else 'falling'
_best_ts = min([('ARIMA', rmse_test, mape_test), ('SARIMA', sarima_rmse, sarima_mape),
                ('Walk-Forward', walk_rmse, walk_mape)], key=lambda x: x[1])
print(f'\n   Takeaway 6 — Prices {direction_word} at ${abs(slope):.0f}/model-year (Phase 2.3 linear trend)')
print(f'      Best time-series model: {_best_ts[0]}  RMSE=${_best_ts[1]:,.0f}  MAPE={_best_ts[2]:.1f}%')
print(f'      Walk-forward MAPE={walk_mape:.1f}% — most realistic estimate for production use')

### 5.3  Model Comparison Visualizations

Side-by-side comparison of every model trained in Phase 4, using held-out test metrics.
Gold bars mark the best performer on each metric.

In [ ]:
# ── Figure 1: Regression models (Phase 4.1) ──────────────────────────────
lb = leaderboard.sort_values('Test R²', ascending=True).reset_index(drop=True)
n  = len(lb)

best_rmse_pos = int(lb['Test RMSE'].argmin())
best_mae_pos  = int(lb['Test MAE'].argmin())

c_r2   = ['gold' if i == n - 1       else 'steelblue' for i in range(n)]
c_rmse = ['gold' if i == best_rmse_pos else 'steelblue' for i in range(n)]
c_mae  = ['gold' if i == best_mae_pos  else 'steelblue' for i in range(n)]

fig, axes = plt.subplots(1, 3, figsize=(17, max(4, n * 0.9 + 1)))

for ax, metric, colors, title in [
    (axes[0], 'Test R²',   c_r2,   'Test R²  (↑ higher = better)'),
    (axes[1], 'Test RMSE', c_rmse, 'Test RMSE  (↓ lower = better)'),
    (axes[2], 'Test MAE',  c_mae,  'Test MAE  (↓ lower = better)'),
]:
    bars = ax.barh(lb['Model'], lb[metric], color=colors, edgecolor='white', height=0.55)
    ax.set_title(title, fontsize=11)
    ax.set_xlabel(metric)
    for bar, val in zip(bars, lb[metric]):
        ax.text(bar.get_width(), bar.get_y() + bar.get_height() / 2,
                f'  {val:.4f}', va='center', fontsize=9)
    ax.set_xlim(0, lb[metric].max() * 1.15)
    ax.spines[['top', 'right']].set_visible(False)

# Polynomial Ridge reference line on R² panel
axes[0].axvline(poly_r2, color='crimson', linestyle='--', linewidth=1.5,
                label=f'Ridge + Poly  {poly_r2:.4f}')
axes[0].legend(fontsize=8, loc='lower right')

plt.suptitle('Regression Model Comparison — Phase 4.1', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('images/model_comparison_regression.png', dpi=150)
plt.show()

best_reg_name = leaderboard.iloc[0]['Model']
print(f'Best regression model: {best_reg_name}  '
      f'R²={leaderboard.iloc[0]["Test R²"]:.4f}  '
      f'RMSE={leaderboard.iloc[0]["Test RMSE"]:.4f}')

# ── Figure 2: Time-series models (Phase 4.3) ─────────────────────────────
ts_labels = [
    f'ARIMA\n{best_arima_order}',
    f'SARIMA\n{best_sarima_order[0]}×{best_sarima_order[1]}',
    'Walk-Forward\nARIMA',
]
ts_rmse_vals = [rmse_test, sarima_rmse, walk_rmse]
ts_mape_vals = [mape_test, sarima_mape, walk_mape]

best_rmse_i = int(np.argmin(ts_rmse_vals))
best_mape_i = int(np.argmin(ts_mape_vals))

c_ts_rmse = ['gold' if i == best_rmse_i else 'steelblue'  for i in range(3)]
c_ts_mape = ['gold' if i == best_mape_i else 'darkorange' for i in range(3)]

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

for ax, vals, colors, ylabel, title, fmt in [
    (axes[0], ts_rmse_vals, c_ts_rmse, 'RMSE ($)',  'Test RMSE  (↓ lower = better)', '${:,.0f}'),
    (axes[1], ts_mape_vals, c_ts_mape, 'MAPE (%)',  'Test MAPE  (↓ lower = better)', '{:.1f}%'),
]:
    bars = ax.bar(ts_labels, vals, color=colors, edgecolor='white', width=0.45)
    ax.set_ylabel(ylabel)
    ax.set_title(title, fontsize=11)
    ax.spines[['top', 'right']].set_visible(False)
    for bar, val in zip(bars, vals):
        ax.text(bar.get_x() + bar.get_width() / 2,
                bar.get_height() + max(vals) * 0.01,
                fmt.format(val), ha='center', va='bottom', fontsize=10, fontweight='bold')
    ax.set_ylim(0, max(vals) * 1.18)

plt.suptitle('Time-Series Model Comparison — Phase 4.3', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('images/model_comparison_timeseries.png', dpi=150)
plt.show()

best_ts_name = ts_labels[best_rmse_i].replace('\n', ' ')
print(f'Best time-series model: {best_ts_name}  '
      f'RMSE=${ts_rmse_vals[best_rmse_i]:,.0f}  '
      f'MAPE={ts_mape_vals[best_rmse_i]:.1f}%')

### 5.2  Overall Summary

In [ ]:
print('=' * 60)
print('  USED VEHICLE PRICE ANALYSIS — OVERALL SUMMARY')
print('=' * 60)

# ── Data ──
print('\n DATA')
print(f'   Raw listings        : 426,000+')
print(f'   After cleaning      : {len(df):,}  (column-by-column missing data strategy)')
print(f'   Features built      : {len(feature_cols)}  ({len(numeric_features)} numeric, {len(label_features)} ordered, {len(te_features)} target-encoded, {len(ohe_names)} one-hot)')

# ── Price prediction ──
print('\n PRICE PREDICTION (Phase 4.1)')
for _, row in leaderboard.iterrows():
    print(f'   {row["Model"]:30s}  Test R² = {row["Test R²"]:.4f}  RMSE = {row["Test RMSE"]:.4f}')
best = leaderboard.iloc[0]
print(f'\n   Best model : {best["Model"]}')
print(f'   Explains {best["Test R²"]*100:.1f}% of price variation.')
print(f'   Remaining {(1-best["Test R²"])*100:.1f}% = factors not in the listing (negotiation, hidden damage, local demand).')

# ── Top drivers ──
print('\n TOP PRICE DRIVERS (Phase 4.1 — Lasso coefficients)')
for _, row in coef_df.head(5).iterrows():
    direction = 'raises' if row['Coefficient'] > 0 else 'lowers'
    print(f'   • {row["Feature"]:30s}  {direction} price  (coef = {row["Coefficient"]:+.3f})')

# ── Segments ──
print('\n MARKET SEGMENTS (Phase 4.2)')
largest_seg = profile['listing_count'].idxmax()
premium_seg = profile['price'].idxmax()
budget_seg  = profile['price'].idxmin()
price_gap   = profile.loc[premium_seg, 'price'] - profile.loc[budget_seg, 'price']
for seg, row in profile.iterrows():
    print(f'   Segment {seg}: avg ${row["price"]:,.0f}  |  '
          f'{row["vehicle_age"]:.0f} yr  |  {row["odometer"]:,.0f} mi  |  '
          f'{int(row["listing_count"]):,} listings')
print(f'   Price gap (premium vs budget) : ${price_gap:,.0f}')
print(f'   Largest segment by volume     : Segment {largest_seg}  ({int(profile.loc[largest_seg, "listing_count"]):,} listings)')

# ── Time-series (exploratory, Phase 2.3) ──
print('\n PRICE TREND — EXPLORATORY (Phase 2.3, median price by model year)')
print(f'   Linear trend slope : ${slope:+.2f} per model year')
print(f'   Trend R²           : {r**2:.4f}  ({r**2*100:.1f}% of cross-year variation explained)')
print(f'   Stationarity (ADF) : {"Stationary" if stationary else "Non-stationary"}  (p={p_val:.4f})')
print(f'   Simple ARIMA order : {order}  →  3-year median price forecast generated')

# ── Time-series modeling (Phase 4.3) ──
print('\n TIME-SERIES MODELING (Phase 4.3 — mean price, ARIMA / SARIMA)')
print(f'   Best ARIMA  : {best_arima_order}  (AIC={best_arima_aic:.2f})  RMSE=${rmse_test:,.0f}  MAPE={mape_test:.2f}%')
print(f'   Best SARIMA : {best_sarima_order}  (AIC={best_sarima_aic:.2f})  RMSE=${sarima_rmse:,.0f}  MAPE={sarima_mape:.2f}%')
print(f'   Walk-forward: RMSE=${walk_rmse:,.0f}  MAPE={walk_mape:.2f}%  (most realistic for production)')
_ts_best = min([('ARIMA', rmse_test), ('SARIMA', sarima_rmse), ('Walk-forward', walk_rmse)], key=lambda x: x[1])
print(f'   Best by RMSE: {_ts_best[0]}  (${_ts_best[1]:,.0f})')

# ── Key takeaways ──
print('\n KEY TAKEAWAYS (all computed from data)')

top2 = coef_df.head(2)['Feature'].tolist()
print(f'   1. Strongest predictors: {top2[0]}  and  {top2[1]}  (highest Lasso |coef|)')

ridge_r2 = leaderboard[leaderboard['Model'].str.startswith('Ridge')]['Test R²'].values
lasso_r2 = leaderboard[leaderboard['Model'].str.startswith('Lasso (')]['Test R²'].values
if len(ridge_r2) and len(lasso_r2):
    gap = abs(float(ridge_r2[0]) - float(lasso_r2[0]))
    print(f'   2. Ridge vs Lasso R² gap = {gap:.4f} — regularisation type barely matters at this data size.')

n_ohe_total = len(ohe_names)
ohe_mask    = [f in ohe_names for f in feature_cols]
ohe_coefs   = lasso.coef_[[i for i, m in enumerate(ohe_mask) if m]]
n_ohe_cut   = int((ohe_coefs == 0).sum())
print(f'   3. Lasso zeroed {n_ohe_cut}/{n_ohe_total} one-hot features ({n_ohe_cut/n_ohe_total*100:.1f}%) — most category columns are noise.')

budget_count  = int(profile.loc[budget_seg,  'listing_count'])
premium_count = int(profile.loc[premium_seg, 'listing_count'])
vol_ratio     = budget_count / premium_count
print(f'   4. Budget segment has {vol_ratio:.1f}x more listings than premium ({budget_count:,} vs {premium_count:,}).')

direction_word = 'rising' if slope > 0 else 'falling'
print(f'   5. Prices are {direction_word} at ${abs(slope):.0f}/yr (Phase 2.3 linear trend on median price).')

print(f'   6. Best time-series model: {_ts_best[0]}  RMSE ${_ts_best[1]:,.0f} — walk-forward is the most production-realistic estimate.')

---
## Phase 6 — Deployment

Translate findings into concrete recommendations and identify what is needed
to move this analysis into a production pricing tool.

### 6.1  Recommendations & Next Steps

In [ ]:
print('=' * 60)
print('  NEXT STEPS & RECOMMENDATIONS')
print('=' * 60)

best_r2     = best_model_row['Test R²']
unexplained = round((1 - best_r2) * 100, 1)

# One-hot noise fraction (low-cardinality features only — manufacturer/state now target-encoded)
n_zeroed  = int((lasso.coef_ == 0).sum())
n_total_f = len(feature_cols)

# Segment price spread
price_gap = profile.loc[premium_seg, 'price'] - profile.loc[budget_seg, 'price']

print(f"""
WHAT THE RESULTS TELL US TO DO NEXT

1. Close the {unexplained}% of unexplained price variation.
   The best model (Test R² = {best_r2}) misses {unexplained}% of price variation.
   Likely causes: seller motivation, local demand, unreported damage.
   Adding zip-code-level features (median income, local inventory)
   is the highest-leverage data improvement available.

2. {n_zeroed} of {n_total_f} features zeroed by Lasso ({n_zeroed/n_total_f*100:.0f}%) — explore interactions.
   Many one-hot fuel/transmission/type columns carry little weight.
   Next step: add cross-product terms between the top predictors
   (e.g. age × condition, age × title_status) to capture the cases
   where one variable changes how much another matters.

3. The ${price_gap:,.0f} gap between budget and premium segments
   suggests a mid-market sweet spot worth studying further.
   Segment {largest_seg} (largest by volume) sits in the middle —
   understanding what moves buyers between segments could inform
   inventory purchasing and pricing strategy for the dealership.

4. ARIMA order {order} was chosen automatically based on the ADF test.
   As more months of data accumulate, re-run the stationarity test
   and consider seasonal decomposition if annual patterns emerge
   (e.g. summer buying season, tax-refund spikes in spring).

5. The model predicts list price, not sale price.
   If listing outcome data (sold / expired / relisted) were available,
   a classification model could predict sale probability — more
   actionable than price alone for a dealership setting.

6. Apply target encoding with cross-validation splits.
   Current target encoding is fit on the full dataset before the
   train/test split, which introduces mild data leakage. A production
   version should encode inside each CV fold for unbiased estimates.
""")